# Processing Katydid Spectrograms

Measures six acoustic parameters from every downloaded katydid recording:
element length, inter-element interval, inter-burst interval, elements per
burst, and its min/max. Unlike the cricket pipeline, this one generates its
own oscillogram from the audio instead of using SINA's spectrogram images--see the README for why.

Stores the result as `katydid_final` for `Display_Function.ipynb`.

In [ ]:
import re
import statistics
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# Root directory: one subfolder per species, holding both the audio downloaded by
# Webscraping.ipynb and the oscillograms this notebook generates from it.
KATYDIDS_DIR = Path.home() / 'Discrete_Signals' / 'Katydids'

## Constants

In [ ]:
# Height fraction excluded around the centerline when measuring ink. Thin, so
# quiet near-centerline elements still register; 0.20 is a tuned compromise
CENTER_EXCLUDE_FRACTION = 0.20

SAVE_DPI = 150

# Generated image width = duration * this, so pixel_time stays fine enough to
# resolve fast trills regardless of recording length.
TARGET_PIXELS_PER_SECOND = 600

MIN_FIGURE_WIDTH_INCHES = 4   # floor so very short clips aren't unreadably small

# Silence kept on each side of the first/last pulse when cropping lead-in/out.
CROP_PAD_SECONDS = 0.15


## Helper Functions

In [ ]:
def gaussian_filter1d(signal, sigma):
    """Apply a 1-D Gaussian smoothing kernel to a 1-D array."""
    kernel_radius    = int(4 * sigma + 0.5)
    kernel_positions = np.arange(-kernel_radius, kernel_radius + 1, dtype=float)
    kernel           = np.exp(-0.5 * (kernel_positions / sigma) ** 2)
    kernel          /= kernel.sum()
    return np.convolve(signal, kernel, mode="same")


def silhouette(values, cluster_labels):
    """Mean silhouette score for a 1-D binary clustering."""
    total_score = 0.0
    for i, value in enumerate(values):
        same_cluster  = values[cluster_labels == cluster_labels[i]]
        other_cluster = values[cluster_labels != cluster_labels[i]]
        within_dist   = np.mean(np.abs(same_cluster  - value)) if len(same_cluster)  > 1 else 0.0
        between_dist  = np.mean(np.abs(other_cluster - value)) if len(other_cluster) > 0 else 0.0
        denom         = max(within_dist, between_dist)
        total_score  += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(values)


def kmeans2(values):
    """Best 2-way split of a 1-D array by exhaustive search over split points.
    Returns (cluster_labels, cluster_centers); label 0 = short, 1 = long."""
    sorted_values    = np.sort(values)
    best_silhouette  = -2.0
    best_split_index = 1

    for i in range(1, len(sorted_values)):
        if i > 1 and sorted_values[i] == sorted_values[i - 1]:
            continue
        split_threshold  = (sorted_values[i - 1] + sorted_values[i]) / 2
        cluster_labels   = (values > split_threshold).astype(int)
        if len(set(cluster_labels)) < 2:
            continue
        split_silhouette = silhouette(values, cluster_labels)
        if split_silhouette > best_silhouette:
            best_silhouette  = split_silhouette
            best_split_index = i

    split_threshold = (sorted_values[best_split_index - 1] + sorted_values[best_split_index]) / 2
    cluster_labels  = (values > split_threshold).astype(int)
    cluster_centers = np.array([
        values[cluster_labels == 0].mean() if (cluster_labels == 0).any() else sorted_values[0],
        values[cluster_labels == 1].mean() if (cluster_labels == 1).any() else sorted_values[-1],
    ])
    return cluster_labels, cluster_centers


def choose_k(gap_durations):
    """1 if the gaps form one cluster, 2 if they split cleanly (silhouette > 0.3)."""
    gap_durations = np.asarray(gap_durations, dtype=float)
    if len(gap_durations) < 3 or len(np.unique(gap_durations)) < 2:
        return 1
    cluster_labels, _ = kmeans2(gap_durations)
    if len(set(cluster_labels)) < 2:
        return 1
    return 2 if silhouette(gap_durations, cluster_labels) > 0.3 else 1


def rle(signal_list):
    """Run-length encode a 1-D sequence."""
    run_list      = []
    current_value = signal_list[0]
    run_length    = 1
    for value in signal_list[1:]:
        if value == current_value:
            run_length += 1
        else:
            run_list.append((current_value, run_length))
            current_value = value
            run_length    = 1
    run_list.append((current_value, run_length))
    return run_list

## Spectrogram Generation

Same idea as the frog pipeline: turn each katydid's downloaded audio into a
clean oscillogram PNG. Crop out lead-in/lead-out silence first (but keep any
inter-burst gaps--those are what we're measuring), find the dominant
frequency, bandpass filter around it, and save a filled waveform with no axes.

Image width scales with the cropped duration, so resolution isn't wasted on
silence and stays fine enough to resolve closely-spaced chirps even in long
recordings.

In [ ]:
def audio_id_from_filename(filename):
    """'Aglaothorax_khioneos_audio_sound.mp3' -> 'sound' (falls back to the stem)."""
    match = re.search(r'_audio_(.+)\.[^.]+$', str(filename))
    return match.group(1) if match else Path(filename).stem


def generated_spectrogram_path(audio_path):
    """Where a katydid's generated oscillogram PNG lives, derived from its audio
    path so the processing function and the batch loop always agree:
    '..._audio_sound.mp3' -> '..._generated_spectrogram_sound.png'."""
    audio_path     = Path(audio_path)
    species_folder = audio_path.parent
    species_name   = species_folder.name
    file_id        = audio_id_from_filename(audio_path.name)
    return species_folder / f"{species_name}_generated_spectrogram_{file_id}.png"


def find_dominant_frequency(signal, sample_rate):
    """Dominant FFT frequency of the clip above 500 Hz (the floor rejects mains
    hum and low-frequency environmental noise)."""
    spectrum  = np.abs(np.fft.rfft(signal))
    freqs     = np.fft.rfftfreq(len(signal), d=1 / sample_rate)
    above_500 = freqs > 500
    peak_idx  = np.argmax(spectrum[above_500])
    return freqs[above_500][peak_idx]


def compute_signal_crop(signal, sample_rate, pad_seconds=CROP_PAD_SECONDS):
    """Sample range from the first to the last audible pulse, padded by
    pad_seconds. Trims only lead-in/lead-out silence--inter-burst gaps between
    the first and last pulse are the signal being measured. The threshold is
    anchored to the noise floor alone (not the eventual peak), so quiet onset
    pulses on crescendo calls survive the crop. Falls
    back to the full signal when there's too little contrast to find an edge.
    Returns (start_sample, end_sample)."""
    frame_length = 2048
    hop_length   = 512
    rms = librosa.feature.rms(y=signal, frame_length=frame_length, hop_length=hop_length)[0]

    noise_floor = np.percentile(rms, 10)
    if noise_floor <= 0:
        return 0, len(signal)

    threshold     = noise_floor * 2.0
    active_frames = np.where(rms >= threshold)[0]
    if not len(active_frames):
        return 0, len(signal)

    pad_samples  = int(pad_seconds * sample_rate)
    start_sample = max(0, active_frames[0] * hop_length - pad_samples)
    end_sample   = min(len(signal), active_frames[-1] * hop_length + frame_length + pad_samples)
    return start_sample, end_sample


def generate_spectrogram_image(signal, sample_rate, output_path, band_width_hz=500):
    """Bandpass-filter an already-cropped katydid clip around its dominant
    frequency (+/- band_width_hz; species-tunable via BAND_WIDTH_HZ_OVERRIDES)
    and save a filled, axis-free oscillogram PNG. Figure width scales with
    duration (TARGET_PIXELS_PER_SECOND) so pixel_time stays fine for dense trills."""
    duration_seconds = len(signal) / sample_rate

    dominant_freq = find_dominant_frequency(signal, sample_rate)

    stft_matrix = librosa.stft(signal)
    magnitude   = np.abs(stft_matrix)
    phase       = np.angle(stft_matrix)
    frequencies = librosa.fft_frequencies(sr=sample_rate)

    freq_mask          = (
        (frequencies >= dominant_freq - band_width_hz) &
        (frequencies <= dominant_freq + band_width_hz)
    )
    magnitude_filtered = magnitude * freq_mask[:, np.newaxis]

    # Suppress the quietest 80% of the filtered magnitude (remove residual noise)
    noise_floor = np.percentile(magnitude_filtered[magnitude_filtered > 0], 80)
    magnitude_filtered[magnitude_filtered < noise_floor] = 0

    filtered_signal = librosa.istft(magnitude_filtered * np.exp(1j * phase))

    time_axis = np.linspace(0, len(filtered_signal) / sample_rate, len(filtered_signal))

    figure_width_inches = max(MIN_FIGURE_WIDTH_INCHES, duration_seconds * TARGET_PIXELS_PER_SECOND / SAVE_DPI)

    fig, ax = plt.subplots(figsize=(figure_width_inches, 2))
    ax.plot(time_axis, filtered_signal, color='black', linewidth=0.5)
    ax.fill_between(time_axis, filtered_signal, alpha=1.0, color='black')
    # Pin the x-axis to [0, duration]; otherwise matplotlib's autoscale margin
    # is baked into the PNG and skews every downstream pixel_time.
    ax.set_xlim(0, duration_seconds)
    ax.axis('off')
    plt.tight_layout(pad=0)

    fig.savefig(str(output_path), dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

## Signal Analysis

A few things tuned specifically for katydids:

- Ink excludes only a thin band around the detected centerline, not a wide
  one--an earlier, wider exclusion band silently dropped real quiet elements.
- Smoothing is narrower than the frog pipeline's (`sigma = 0.15`), since a
  wider kernel started blurring out real short gaps once resolution went up.
- 1-pixel gaps get filled only in dense trills, so real noise blips don't
  fracture one continuous trill into several fake elements.
- The active-fraction ceiling is a bit looser than crickets'--some katydid
  trills genuinely fill more of the recording.
- Thresholding tries a standard ladder first, then an extended one for
  high-baseline recordings, then falls back to treating the whole thing as one
  continuous trill. See `detect_signal_list_adaptive`'s docstring.

In [ ]:
def clean_signal_runs(signal_list, pixel_time, fill_gap_size=0):
    """Drop on-runs shorter than 3 ms. With fill_gap_size > 0, also fill
    off-gaps up to that wide between two on-runs (used for dense trills where a
    1-pixel gap may just be image noise)."""
    signal_list   = np.asarray(signal_list, dtype=np.uint8)
    min_on_pixels = max(1, round(0.003 / pixel_time))

    cleaned = []
    for value, run_length in rle(signal_list):
        if value == 1 and run_length < min_on_pixels:
            cleaned.extend([0] * run_length)
        else:
            cleaned.extend([int(value)] * run_length)
    cleaned = np.array(cleaned, dtype=np.uint8)

    if fill_gap_size > 0 and len(cleaned) > 2:
        result_list = []
        run_list    = rle(cleaned)
        for i, (value, run_length) in enumerate(run_list):
            surrounded_by_signal = (
                i > 0
                and i < len(run_list) - 1
                and run_list[i - 1][0] == 1
                and run_list[i + 1][0] == 1
            )
            if value == 0 and surrounded_by_signal and run_length <= fill_gap_size:
                result_list.extend([1] * run_length)
            else:
                result_list.extend([int(value)] * run_length)
        cleaned = np.array(result_list, dtype=np.uint8)

    return cleaned


def find_centerline_row(spec_array):
    """Row of the always-dark zero-amplitude baseline, found by darkness rather
    than assumed to be the image's middle (a tight bbox can crop asymmetrically)."""
    row_dark_fraction = (spec_array < 200).mean(axis=1)
    return int(np.argmax(row_dark_fraction))


def extract_outer_band_ink(spec_array, center_exclude_fraction=None):
    """2-D ink array with only a thin band around the centerline excluded, so
    quiet near-baseline elements still register. center_exclude_fraction
    overrides the global default (see CENTER_EXCLUDE_FRACTION_OVERRIDES)."""
    if center_exclude_fraction is None:
        center_exclude_fraction = CENTER_EXCLUDE_FRACTION
    img_height, img_width = spec_array.shape
    center_row   = find_centerline_row(spec_array)
    half_exclude = max(1, int(img_height * center_exclude_fraction / 2))
    rows         = np.arange(img_height)
    keep_mask    = np.abs(rows - center_row) > half_exclude
    band         = spec_array[keep_mask, :].astype(float)

    # Ink = darkness relative to background (white background -> near-zero ink in gaps)
    background_level = np.percentile(band, 95)
    ink              = np.clip(background_level - band, 0, None)
    ink_peak         = np.percentile(ink, 99)
    if ink_peak > 0:
        ink /= ink_peak

    return ink


def detect_signal_list_adaptive(ink_array, pixel_time, fill_gap_size_override=None):
    """Threshold the ink array into a binary on/off signal. Walks a standard
    ladder, then an extended high-baseline one, then falls back to a single
    continuous on-run. Returns (signal_list, normalized_signal)."""
    col_pct85           = np.percentile(ink_array, 85, axis=0)
    col_pct95           = np.percentile(ink_array, 95, axis=0)
    col_pct99           = np.percentile(ink_array, 99, axis=0)
    col_signal_strength = gaussian_filter1d(
        0.20 * col_pct85 + 0.35 * col_pct95 + 0.45 * col_pct99,
        sigma=0.15  # narrowed from 0.45 
    )

    signal_floor      = np.percentile(col_signal_strength,  5)
    signal_peak       = np.percentile(col_signal_strength, 99.5)
    if signal_peak <= signal_floor:
        raise ValueError("No contrast in ink array")
    normalized_signal = np.clip(
        (col_signal_strength - signal_floor) / (signal_peak - signal_floor), 0, 1
    )

    min_on_pixels = max(1, round(0.003 / pixel_time))
    # Fill 1-pixel off-gaps only for fine-grained images (dense trills), unless a
    # species-specific override is given (see FILL_GAP_SIZE_OVERRIDES)
    fill_gap_size = fill_gap_size_override if fill_gap_size_override is not None else (
        1 if pixel_time <= 0.004 else 0
    )

    def try_ladder(ladder):
        candidates = []
        for high_threshold, low_threshold in ladder:
            high_mask = normalized_signal >= high_threshold
            low_mask  = normalized_signal >= low_threshold

            signal_list = np.zeros_like(low_mask, dtype=np.uint8)
            column_pos  = 0
            for value, run_length in rle(low_mask.astype(np.uint8)):
                run_start, run_end = column_pos, column_pos + run_length
                if value == 1 and run_length >= min_on_pixels and np.any(high_mask[run_start:run_end]):
                    signal_list[run_start:run_end] = 1
                column_pos = run_end

            signal_list     = clean_signal_runs(signal_list, pixel_time, fill_gap_size)
            active_fraction = float(signal_list.mean())

            if active_fraction <= 0 or active_fraction >= 0.95:
                continue

            on_run_lengths = [run_length for value, run_length in rle(signal_list) if value == 1]
            if not on_run_lengths:
                continue

            median_on_length  = float(np.median(on_run_lengths))
            tiny_run_fraction = sum(l <= 1 for l in on_run_lengths) / len(on_run_lengths)
            score = high_threshold - 0.15 * tiny_run_fraction - 0.002 * median_on_length
            candidates.append((score, signal_list))
        return candidates

    standard_ladder = [(t, max(0.06, t * 0.45)) for t in
                        [0.78, 0.70, 0.62, 0.54, 0.46, 0.38, 0.30, 0.22, 0.16, 0.10]]
    candidate_thresholds = try_ladder(standard_ladder)

    if not candidate_thresholds:
        extended_ladder = [(t, max(0.06, t - 0.12)) for t in [0.98, 0.94, 0.90, 0.86, 0.82]]
        candidate_thresholds = try_ladder(extended_ladder)

    if not candidate_thresholds:
        # Every threshold in both ladders came back too dense (>= 0.95 active):
        # a genuine continuous trill, not a detection failure. Fall back to one
        # long on-run at a low threshold with no active-fraction ceiling.
        low_threshold = 0.10
        low_mask      = normalized_signal >= low_threshold
        signal_list   = clean_signal_runs(low_mask.astype(np.uint8), pixel_time, fill_gap_size)
        if signal_list.mean() <= 0:
            raise ValueError("No valid signal detected at any threshold")
        return signal_list, normalized_signal

    candidate_thresholds.sort(key=lambda t: t[0], reverse=True)
    return candidate_thresholds[0][1], normalized_signal


def group_consecutive(indices, max_gap_px):
    """Group sorted column indices into contiguous (start, end) regions, merging
    across gaps of at most max_gap_px."""
    if not len(indices):
        return []
    regions = []
    start = prev = indices[0]
    for idx in indices[1:]:
        if idx - prev <= max_gap_px:
            prev = idx
        else:
            regions.append((start, prev))
            start = prev = idx
    regions.append((start, prev))
    return regions


def rescue_quiet_bursts(spec_array, ink, raw_signal_list, pixel_time,
                         center_exclude_fraction=None,
                         presence_threshold=0.03, max_gap_px=8,
                         min_region_px=4, pad_px=3):
    """Recover bursts real but too quiet (relative to a louder burst in the same
    recording) to clear the global threshold--ink is normalized from
    whole-image percentiles, so a loud burst can crush a quiet one toward 0.
    Re-crops each missed region and re-runs detection locally. See
    . Returns raw_signal_list OR'd with anything rescued."""
    combined = raw_signal_list.copy()
    presence_cols = np.where(ink.max(axis=0) > presence_threshold)[0]
    missed_cols = presence_cols[combined[presence_cols] == 0]
    if not len(missed_cols):
        return combined

    for region_start, region_end in group_consecutive(missed_cols, max_gap_px):
        if region_end - region_start + 1 < min_region_px:
            continue

        pad_start = max(0, region_start - pad_px)
        pad_end   = min(spec_array.shape[1], region_end + pad_px + 1)

        # Already-detected columns touching this region's padded edges mean the
        # global pass DID find something right next to it -- likely this is that
        # same burst's quiet onset/tail, already counted, not a separate burst.
        if combined[pad_start:pad_end].any():
            continue

        crop = spec_array[:, pad_start:pad_end]
        try:
            local_ink = extract_outer_band_ink(crop, center_exclude_fraction=center_exclude_fraction)
            local_signal, _ = detect_signal_list_adaptive(local_ink, pixel_time)
        except ValueError:
            continue  # local region too flat/small to threshold at all -- leave undetected

        combined[pad_start:pad_end] = np.maximum(combined[pad_start:pad_end], local_signal)

    return combined


## Interval Classification

In [ ]:
def classify_intervals(time_bucket_list, leading_silence, trailing_silence,
                       element_length, pixel_time, force_trill=False, min_gap_ratio=5.0):
    """Classify off-run durations as inter-element vs. inter-burst by clustering
    them and checking guards (gap ratio, separation, position, pixel width);
    . min_gap_ratio (default 5.0) is relaxed per species
    via RELAXED_BURST_GAP_RATIO_SPECIES; force_trill skips clustering for
    CONTINUOUS_TRILL_SPECIES. Returns (inter_element_interval,
    inter_burst_interval, elements_per_burst, min_elements_per_burst,
    max_elements_per_burst, burst_gap_ratio); no burst structure collapses all
    three counts to the same value."""
    off_durations = np.array(
        [duration for state, duration in time_bucket_list if state == "Off"],
        dtype=float,
    )

    if len(off_durations) == 0:
        edge_gaps = [x for x in [leading_silence, trailing_silence] if x > 0]
        return 0.0, float(np.mean(edge_gaps)) if edge_gaps else 0.0, 1, 1, 1, 0.0

    if force_trill:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1, 0.0

    if len(off_durations) < 4 or len(np.unique(np.round(off_durations, 6))) < 2:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1, 0.0

    if choose_k(off_durations) != 2:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1, 0.0

    cluster_labels, cluster_centers = kmeans2(off_durations)
    short_gap_durations = off_durations[cluster_labels == 0]
    long_gap_durations  = off_durations[cluster_labels == 1]
    inter_element_mean  = float(np.mean(short_gap_durations))
    inter_burst_mean    = float(np.mean(long_gap_durations))

    if inter_element_mean <= 0:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1, 0.0

    burst_gap_ratio     = inter_burst_mean / inter_element_mean
    absolute_separation = inter_burst_mean - inter_element_mean
    long_gap_fraction   = len(long_gap_durations) / len(off_durations)
    n_long_gaps         = len(long_gap_durations)
    inter_burst_pixels  = inter_burst_mean / pixel_time

    if n_long_gaps == 1:
        single_long_gap_index = int(np.where(cluster_labels == 1)[0][0])
        single_gap_position   = single_long_gap_index / max(1, len(off_durations) - 1)
    else:
        single_gap_position = 0.5

    is_burst = (
        burst_gap_ratio     >= min_gap_ratio  # default 5.0, raised from 4.0 for katydids
        and inter_burst_mean    >= 2.5 * element_length
        and absolute_separation >= 1.5 * element_length
        and long_gap_fraction   <= 0.55
        and inter_burst_pixels  >= 10
        and (
            n_long_gaps >= 2
            or (inter_burst_pixels >= 200 and single_gap_position >= 0.25)
        )
    )

    # 3-cluster fallback: when only one gap is long and it's near the recording's edge
    # (leading/trailing silence, not a real between-verse gap by this guard's logic),
    # check whether the short cluster itself splits into a second burst structure.
    if not is_burst and n_long_gaps == 1 and len(short_gap_durations) >= 6:
        if choose_k(short_gap_durations) == 2:
            secondary_labels, _ = kmeans2(short_gap_durations)
            secondary_silhouette = silhouette(short_gap_durations, secondary_labels)
            secondary_short = short_gap_durations[secondary_labels == 0]
            secondary_long  = short_gap_durations[secondary_labels == 1]
            if (secondary_silhouette > 0.5 and len(secondary_long) >= 2
                    and float(np.mean(secondary_long)) >= 4.0 * float(np.mean(secondary_short))
                    and float(np.mean(secondary_long)) / pixel_time >= 5):
                inter_element_mean = float(np.mean(secondary_short))
                inter_burst_mean   = float(np.mean(secondary_long))
                burst_gap_ratio    = inter_burst_mean / inter_element_mean
                long_gap_fraction  = len(secondary_long) / len(short_gap_durations)
                inter_burst_pixels = inter_burst_mean / pixel_time
                n_long_gaps        = len(secondary_long)
                is_burst = (
                    burst_gap_ratio    >= min_gap_ratio
                    and inter_burst_mean   >= 2.5 * element_length
                    and long_gap_fraction  <= 0.55
                    and inter_burst_pixels >= 10
                    and n_long_gaps        >= 2
                )

    if not is_burst:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1, burst_gap_ratio

    burst_boundary_time     = (inter_element_mean + inter_burst_mean) / 2
    elements_per_burst_list = []
    current_count           = 0
    for state, duration in time_bucket_list:
        if state == "On":
            current_count += 1
        elif duration > burst_boundary_time:
            if current_count > 0:
                elements_per_burst_list.append(current_count)
            current_count = 0
    if current_count > 0:
        elements_per_burst_list.append(current_count)

    median_elements_per_burst = (
        int(statistics.median(elements_per_burst_list))
        if elements_per_burst_list else 1
    )
    min_elements_per_burst = min(elements_per_burst_list) if elements_per_burst_list else 1
    max_elements_per_burst = max(elements_per_burst_list) if elements_per_burst_list else 1
    return (
        inter_element_mean, inter_burst_mean, max(1, median_elements_per_burst),
        min_elements_per_burst, max_elements_per_burst, burst_gap_ratio,
    )


## Audio Cropping for Long / Ambiguous Recordings

A few species have recordings so long, or with so many different scales of
gap (element, burst, verse), that the usual two-cluster gap detection can't
tell inter-element from inter-burst gaps anymore--it'll happily report an
inter-element gap of nearly a second, which is nonsense.

For a known list of species (plus anything that produces a similarly
degenerate result), we crop the audio down to 2-3 clean bursts before ever
generating the oscillogram--same idea as `Frog_Clip_Log.ipynb`, done inline
here instead of as a separate notebook. If no clean burst structure turns up,
it falls back to just grabbing the first 15 elements.

Cropped clips go to `Cropped_Katydids_Audios/`, logged the same way
`frog_clip_log.csv` is, and get substituted in wherever `katydid_process`
would otherwise load the original file.

In [ ]:
# Root folder for manually-cropped audio clips + crop log (mirrors Cropped_Frogs_Audios/).
CROPPED_KATYDIDS_DIR = Path.home() / 'Discrete_Signals' / 'Cropped_Katydids_Audios'
CROPPED_KATYDIDS_DIR.mkdir(exist_ok=True)

# Recordings long/multi-scale enough that bimodal burst detection breaks down;
# cropped to a 2-3 burst window before the oscillogram is generated.
LONG_RECORDING_SPECIES = [
    'Amblycorypha_longinicta', 'Amblycorypha_rivograndis', 'Insara_covilleae',
    'Amblycorypha_rotundifolia', 'Atlanticus_americanus', 'Atlanticus_calcaratus',
    'Atlanticus_dorsalis', 'Atlanticus_gibbosus', 'Atlanticus_monticola',
    'Bucrates_malivolans', 'Capnobotes_bruneri', 'Capnobotes_fuliginosus',
    'Conocephalus_brevipennis', 'Conocephalus_fasciatus', 'Eremopedes_balli',
    'Neduba_arborea', 'Neduba_cascadia', 'Neduba_inversa', 'Neduba_longiplutea',
    'Neduba_macneilli', 'Neduba_oblongata', 'Neduba_propsti', 'Neduba_radocantans',
    'Scudderia_furcata',
]

# Confirmed one continuous trill; forces elements_per_burst=1 so the 3-cluster
# fallback can't invent a "burst" from amplitude dips.
CONTINUOUS_TRILL_SPECIES = [
    'Idiostatus_hermannii',
    'Neoconocephalus_melanorhinus',
    'Neduba_sierranus',
    'Neduba_oblongata',
    'Orchelimum_gladiator',
    'Neduba_ambagiosa',
    'Neduba_prorocantans',
    'Conocephalus_brevipennis',
]

# Per-species fill_gap_size for clean_signal_runs (default is too small for
# these.).
FILL_GAP_SIZE_OVERRIDES = {
    'Aglaothorax_tinkhamorum': 4,
    'Neduba_lucubrata': 5,
}

# Species-specific override for classify_intervals' burst_gap_ratio guard.
RELAXED_BURST_GAP_RATIO_SPECIES = {
    'Aglaothorax_hulodomus': 3.0,
}

# Per-species CENTER_EXCLUDE_FRACTION; the global 0.20 still drops real quiet
# elements for these.
CENTER_EXCLUDE_FRACTION_OVERRIDES = {
    'Scudderia_cuneata': 0.05,
    'Inscudderia_strigata': 0.03,
    'Inscudderia_taxodii': 0.03,
    'Idionotus_tehachapi': 0.10,
    'Eremopedes_bilineatus': 0.16,
    'Arethaea_phalangium': 0.10,
}

# Species whose only SINA recording isn't a calling song (per katydid_df's
# Description), so it isn't comparable to the rest. Skipped entirely.
EXCLUDED_NON_CALLING_SONG_SPECIES = [
    'Paracyrtophyllus_excelsus',  # katydid_df Description: "14 s of protest song"
]

# Per-species bandpass width for generate_spectrogram_image; the default +/-500 Hz
# cuts into real signal for these.
BAND_WIDTH_HZ_OVERRIDES = {
    'Amblycorypha_huasteca': 2000,
    'Amblycorypha_parvipennis': 3000,
}


def detect_audio_elements(signal, sample_rate, frame_length=512, hop_length=128, min_element_s=0.003):
    """Coarse RMS-level element detector, used only to pick a crop window (finer
    hop than compute_signal_crop so ~10 ms elements resolve). Not the detector
    behind the reported metrics."""
    rms = librosa.feature.rms(y=signal, frame_length=frame_length, hop_length=hop_length)[0]
    noise_floor = np.percentile(rms, 10)
    threshold = np.percentile(rms, 50) * 0.5 if noise_floor <= 0 else noise_floor * 2.0
    active = rms >= threshold

    raw_runs = []
    in_run, run_start = False, 0
    for i, v in enumerate(active):
        if v and not in_run:
            in_run, run_start = True, i
        elif not v and in_run:
            in_run = False
            raw_runs.append((run_start, i))
    if in_run:
        raw_runs.append((run_start, len(active)))

    min_frames = max(1, int(min_element_s * sample_rate / hop_length))
    raw = [
        (s * hop_length, min(len(signal), e * hop_length + frame_length))
        for s, e in raw_runs if e - s >= min_frames
    ]

    # Merge runs separated by a very brief dip (mid-pulse amplitude modulation,
    # not a real inter-element silence) -- otherwise one physical pulse gets
    # fragmented into several spurious "elements".
    merge_gap_samples = int(0.006 * sample_rate)
    merged = []
    for s, e in raw:
        if merged and s - merged[-1][1] <= merge_gap_samples:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))
    return merged


def select_crop_window(elements, sample_rate, total_samples, pad_seconds=0.2,
                        min_elements_fallback=15, max_bursts=3,
                        max_burst_elements=40, max_burst_duration_s=12.0,
                        min_burst_elements=3, dense_passage_min_duration_s=0.3):
    """Pick a (start_sample, end_sample, note) window spanning 2-3 bursts, or the
    first min_elements_fallback elements if there's no clean burst structure
    (checked with the same kmeans2/choose_k gap split as classify_intervals,
    plus plausible-burst-size guards). Returns None when the recording is
    already short enough. Special case: if the coarse detector merged a dense
    trill into one giant element (> dense_passage_min_duration_s), crop straight
    to that element's span."""
    n = len(elements)
    if n == 0:
        return None

    dense_passage_idx = next(
        (i for i, (s, e) in enumerate(elements) if (e - s) / sample_rate >= dense_passage_min_duration_s),
        None,
    )
    if n < min_elements_fallback and dense_passage_idx is not None:
        pad = int(pad_seconds * sample_rate)
        s, e = elements[dense_passage_idx]
        start = max(0, s - pad)
        end   = min(total_samples, e + pad)
        duration_s = (e - s) / sample_rate
        return start, end, f"dense passage window (merged element, {duration_s:.2f}s)"

    gaps = np.array([
        (elements[i + 1][0] - elements[i][1]) / sample_rate
        for i in range(n - 1)
    ])

    if n < min_elements_fallback:
        return None  # already concise -- no crop needed

    burst_threshold = None
    if len(gaps) >= 4 and len(np.unique(np.round(gaps, 6))) >= 2 and choose_k(gaps) == 2:
        labels, _ = kmeans2(gaps)
        short_gaps = gaps[labels == 0]
        long_gaps  = gaps[labels == 1]
        if len(long_gaps) >= 4 and long_gaps.max() / max(long_gaps.min(), 1e-9) > 3 and choose_k(long_gaps) == 2:
            labels2, _ = kmeans2(long_gaps)
            burst_threshold = long_gaps[labels2 == 0].max()
        else:
            burst_threshold = short_gaps.max()

    bursts = [[0]]
    if burst_threshold is not None:
        for i, gap in enumerate(gaps):
            if gap > burst_threshold:
                bursts.append([])
            bursts[-1].append(i + 1)
    n_bursts = len(bursts) if burst_threshold is not None else 0

    if n_bursts >= 2:
        burst_sizes = [len(b) for b in bursts]
        burst_durations = [
            (elements[b[-1]][1] - elements[b[0]][0]) / sample_rate for b in bursts
        ]
        if (max(burst_sizes) > max_burst_elements
                or max(burst_durations) > max_burst_duration_s
                or np.median(burst_sizes) < min_burst_elements):
            n_bursts = 0  # not real burst structure -- use element-count fallback

    pad = int(pad_seconds * sample_rate)
    if n_bursts >= 2:
        chosen = bursts[:max_bursts]
        start = max(0, elements[chosen[0][0]][0] - pad)
        end   = min(total_samples, elements[chosen[-1][-1]][1] + pad)
        return start, end, f"burst window (bursts={len(chosen)} of {n_bursts})"
    else:
        take = min(n, min_elements_fallback) if n >= min_elements_fallback else n
        start = max(0, elements[0][0] - pad)
        end   = min(total_samples, elements[take - 1][1] + pad)
        return start, end, f"element window (elements={take}, no clean burst structure)"


In [ ]:
import soundfile as sf

crop_log_path = CROPPED_KATYDIDS_DIR / 'katydid_crop_log.csv'

crop_rows = []
for species_folder in LONG_RECORDING_SPECIES:
    genus, species = species_folder.split('_', 1)
    sp_dir = KATYDIDS_DIR / species_folder
    audio_files = sorted(
        p for p in sp_dir.glob(f'{species_folder}_audio_*')
        if p.suffix.lower() in {'.mp3', '.wav', '.ogg'}
    )
    for audio_num, audio_path in enumerate(audio_files, start=1):
        raw_signal, sample_rate = librosa.load(str(audio_path), sr=None)
        source_dur_s = len(raw_signal) / sample_rate
        elements = detect_audio_elements(raw_signal, sample_rate)
        result = select_crop_window(elements, sample_rate, len(raw_signal))

        if result is None:
            crop_rows.append(dict(
                genus=genus, species=species, audio_num=audio_num,
                source_file=str(audio_path), clipped_file='',
                status='skipped', note=f'no crop needed ({len(elements)} elements)',
                source_dur_s=round(source_dur_s, 3),
                start_time_s=None, end_time_s=None, clip_duration_s=None,
            ))
            continue

        start, end, note = result
        clip = raw_signal[start:end]
        clipped_file = CROPPED_KATYDIDS_DIR / f'{species_folder}_{audio_num}.wav'
        sf.write(clipped_file, clip, sample_rate)
        crop_rows.append(dict(
            genus=genus, species=species, audio_num=audio_num,
            source_file=str(audio_path), clipped_file=str(clipped_file),
            status='ok', note=note,
            source_dur_s=round(source_dur_s, 3),
            start_time_s=round(start / sample_rate, 3),
            end_time_s=round(end / sample_rate, 3),
            clip_duration_s=round(len(clip) / sample_rate, 3),
        ))

katydid_crop_log = pd.DataFrame(crop_rows, columns=[
    'genus', 'species', 'audio_num', 'source_file', 'clipped_file',
    'status', 'note', 'source_dur_s', 'start_time_s', 'end_time_s', 'clip_duration_s',
])
katydid_crop_log.to_csv(crop_log_path, index=False)

# Lookup used by katydid_process: original audio path (str) -> cropped audio path (str).
# Only present for rows that were actually cropped ('ok'); everything else keeps
# using its original, uncropped audio file.
KATYDID_CROP_LOOKUP = {
    row['source_file']: row['clipped_file']
    for row in crop_rows if row['status'] == 'ok'
}

print(f"{len(katydid_crop_log)} files evaluated | "
      f"{(katydid_crop_log.status == 'ok').sum()} cropped | "
      f"{(katydid_crop_log.status == 'skipped').sum()} left as-is")
katydid_crop_log

In [ ]:
# Store for reference elsewhere (mirrors frog pipeline's clips_ready / %store pattern).
%store katydid_crop_log

## Processing Function

`katydid_process` takes a downloaded audio file, not an image--it generates
the oscillogram itself if needed, works out `pixel_time` from the audio's own
duration, and reuses the same ink/threshold/classify steps as the frog
pipeline.

In [ ]:
def katydid_process(audio_path):
    """Extract the six acoustic parameters from one katydid recording: resolve a
    manual crop if any → crop to the active span → generate/load the oscillogram
    → threshold ink → rescue quiet bursts → trim → classify gaps. Consults
    CENTER_EXCLUDE_FRACTION_OVERRIDES, FILL_GAP_SIZE_OVERRIDES,
    RELAXED_BURST_GAP_RATIO_SPECIES, CONTINUOUS_TRILL_SPECIES."""
    audio_path   = Path(audio_path)
    species_name = audio_path.parent.name

    # Generated PNG name/location is still derived from the ORIGINAL audio_path,
    # so existing filenames are unaffected -- only the audio actually measured changes.
    source_path = Path(KATYDID_CROP_LOOKUP.get(str(audio_path), audio_path))

    raw_signal, sample_rate = librosa.load(str(source_path), sr=None)
    crop_start, crop_end    = compute_signal_crop(raw_signal, sample_rate)
    signal                  = raw_signal[crop_start:crop_end]
    duration_seconds        = len(signal) / sample_rate

    spec_path = generated_spectrogram_path(audio_path)
    if not spec_path.exists():
        generate_spectrogram_image(
            signal, sample_rate, spec_path,
            band_width_hz=BAND_WIDTH_HZ_OVERRIDES.get(species_name, 500),
        )

    spec_array = np.array(Image.open(str(spec_path)).convert('L'))
    if spec_array.shape[1] == 0:
        raise ValueError("Empty image")
    pixel_time = duration_seconds / spec_array.shape[1]

    ink = extract_outer_band_ink(
        spec_array, center_exclude_fraction=CENTER_EXCLUDE_FRACTION_OVERRIDES.get(species_name)
    )

    raw_signal_list, _ = detect_signal_list_adaptive(ink, pixel_time)

    raw_signal_list = rescue_quiet_bursts(
        spec_array, ink, raw_signal_list, pixel_time,
        center_exclude_fraction=CENTER_EXCLUDE_FRACTION_OVERRIDES.get(species_name),
    )

    signal_columns = np.where(raw_signal_list == 1)[0]
    if not len(signal_columns):
        raise ValueError("No signal columns detected")

    trim_start = max(0, signal_columns[0]  - 2)
    trim_end   = min(len(raw_signal_list), signal_columns[-1] + 3)

    leading_silence  = trim_start * pixel_time
    trailing_silence = (len(raw_signal_list) - trim_end) * pixel_time

    signal_list = clean_signal_runs(
        raw_signal_list[trim_start:trim_end], pixel_time,
        fill_gap_size=FILL_GAP_SIZE_OVERRIDES.get(species_name, 0),
    )
    if not len(signal_list) or signal_list.mean() == 0:
        raise ValueError("Empty signal after cleanup")

    time_buckets = [
        ("On" if value == 1 else "Off", run_length * pixel_time)
        for value, run_length in rle(signal_list)
    ]

    on_durations = [duration for state, duration in time_buckets if state == "On"]
    if not on_durations:
        raise ValueError("No on-pulses detected")
    element_length = float(np.mean(on_durations))

    force_trill = species_name in CONTINUOUS_TRILL_SPECIES
    (
        inter_element_interval, inter_burst_interval, elements_per_burst,
        min_elements_per_burst, max_elements_per_burst, _,
    ) = classify_intervals(
        time_buckets, leading_silence, trailing_silence, element_length, pixel_time,
        force_trill=force_trill,
        min_gap_ratio=RELAXED_BURST_GAP_RATIO_SPECIES.get(species_name, 5.0),
    )

    return {
        "element_length":         round(element_length,         4),
        "inter_element_interval": round(inter_element_interval, 4),
        "inter_burst_interval":   round(inter_burst_interval,   4),
        "elements_per_burst":     elements_per_burst,
        "min_elements_per_burst": min_elements_per_burst,
        "max_elements_per_burst": max_elements_per_burst,
    }


## Run Pipeline on All Audio Files

Runs every downloaded katydid audio file through the pipeline and saves
results to `katydid_results.csv`. Species with no audio are skipped with an
error; species on `EXCLUDED_NON_CALLING_SONG_SPECIES` are skipped outright--their only recording isn't a calling song, so there's nothing to measure.

In [ ]:
audio_files = sorted(
    p for p in KATYDIDS_DIR.rglob("*_audio_*")
    if "checkpoint" not in str(p)
    and p.suffix.lower() in {".mp3", ".wav", ".ogg"}
)

rows = []
for audio_path in audio_files:
    species_folder = audio_path.parent.name
    if species_folder in EXCLUDED_NON_CALLING_SONG_SPECIES:
        continue
    spec_path      = generated_spectrogram_path(audio_path)
    try:
        result = katydid_process(audio_path)
        status = "ok"
        error  = ""
    except Exception as e:
        result = {}
        status = "error"
        error  = str(e)
    rows.append({
        # 'file' holds the generated spectrogram's name (not the audio filename) so
        # the webscraping merge step below can keep using its existing '_spectrogram_' regex.
        "species": species_folder,
        "file":    spec_path.name,
        "status":  status,
        "error":   error,
        **result,
    })

output_csv  = KATYDIDS_DIR / "katydid_results.csv"
import csv
csv_columns = [
    "species", "file", "status", "error",
    "element_length", "inter_element_interval",
    "inter_burst_interval", "elements_per_burst",
    "min_elements_per_burst", "max_elements_per_burst",
]
with open(output_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

successful_rows = [r for r in rows if r["status"] == "ok"]
error_rows      = [r for r in rows if r["status"] == "error"]
print(f"Processed {len(rows)} audio files: {len(successful_rows)} ok, {len(error_rows)} errors")
for r in error_rows:
    print(f"  ERROR {r['species']} / {r['file']}: {r['error']}")

## Species-Specific Override: Eremopedes_covilleae

This species has real inter-element gaps as narrow as a single pixel, which
the pipeline's default 1-pixel gap-fill was merging into one element. It
needs the opposite of every other override--gap-filling turned off--so
rather than bolt reversed semantics onto the existing override dicts, it gets
its own function (`process_eremopedes_covilleae`) that patches this species'
rows into `katydid_results.csv` after the main run.

In [ ]:
def process_eremopedes_covilleae(audio_path):
    """Eremopedes_covilleae-only variant of katydid_process: identical except
    detect_signal_list_adaptive runs with fill_gap_size_override=0 (its real
    1-pixel gaps were being merged). See the markdown above."""
    audio_path   = Path(audio_path)
    species_name = audio_path.parent.name  # always 'Eremopedes_covilleae' when called below

    source_path = Path(KATYDID_CROP_LOOKUP.get(str(audio_path), audio_path))

    raw_signal, sample_rate = librosa.load(str(source_path), sr=None)
    crop_start, crop_end    = compute_signal_crop(raw_signal, sample_rate)
    signal                  = raw_signal[crop_start:crop_end]
    duration_seconds        = len(signal) / sample_rate

    spec_path = generated_spectrogram_path(audio_path)
    if not spec_path.exists():
        generate_spectrogram_image(signal, sample_rate, spec_path)

    spec_array = np.array(Image.open(str(spec_path)).convert('L'))
    if spec_array.shape[1] == 0:
        raise ValueError("Empty image")
    pixel_time = duration_seconds / spec_array.shape[1]

    ink = extract_outer_band_ink(
        spec_array, center_exclude_fraction=CENTER_EXCLUDE_FRACTION_OVERRIDES.get(species_name)
    )

    # The one difference from katydid_process: fill_gap_size_override=0 disables the
    # default "fill 1-pixel off-gaps" rule, which was merging this species' genuinely
    # separate elements (see markdown above).
    raw_signal_list, _ = detect_signal_list_adaptive(ink, pixel_time, fill_gap_size_override=0)

    raw_signal_list = rescue_quiet_bursts(
        spec_array, ink, raw_signal_list, pixel_time,
        center_exclude_fraction=CENTER_EXCLUDE_FRACTION_OVERRIDES.get(species_name),
    )

    signal_columns = np.where(raw_signal_list == 1)[0]
    if not len(signal_columns):
        raise ValueError("No signal columns detected")

    trim_start = max(0, signal_columns[0]  - 2)
    trim_end   = min(len(raw_signal_list), signal_columns[-1] + 3)

    leading_silence  = trim_start * pixel_time
    trailing_silence = (len(raw_signal_list) - trim_end) * pixel_time

    signal_list = clean_signal_runs(
        raw_signal_list[trim_start:trim_end], pixel_time,
        fill_gap_size=FILL_GAP_SIZE_OVERRIDES.get(species_name, 0),
    )
    if not len(signal_list) or signal_list.mean() == 0:
        raise ValueError("Empty signal after cleanup")

    time_buckets = [
        ("On" if value == 1 else "Off", run_length * pixel_time)
        for value, run_length in rle(signal_list)
    ]

    on_durations = [duration for state, duration in time_buckets if state == "On"]
    if not on_durations:
        raise ValueError("No on-pulses detected")
    element_length = float(np.mean(on_durations))

    force_trill = species_name in CONTINUOUS_TRILL_SPECIES
    (
        inter_element_interval, inter_burst_interval, elements_per_burst,
        min_elements_per_burst, max_elements_per_burst, _,
    ) = classify_intervals(
        time_buckets, leading_silence, trailing_silence, element_length, pixel_time,
        force_trill=force_trill,
    )

    return {
        "element_length":         round(element_length,         4),
        "inter_element_interval": round(inter_element_interval, 4),
        "inter_burst_interval":   round(inter_burst_interval,   4),
        "elements_per_burst":     elements_per_burst,
        "min_elements_per_burst": min_elements_per_burst,
        "max_elements_per_burst": max_elements_per_burst,
    }


# Run the species-specific variant only for Eremopedes_covilleae, then patch its rows
# into the existing katydid_results.csv in place. katydid_process and the main loop
# above are untouched, so every other species' rows are unaffected.
eremopedes_audio_files = sorted(
    p for p in (KATYDIDS_DIR / 'Eremopedes_covilleae').glob('Eremopedes_covilleae_audio_*')
    if p.suffix.lower() in {'.mp3', '.wav', '.ogg'}
)

eremopedes_rows = []
for audio_path in eremopedes_audio_files:
    spec_path = generated_spectrogram_path(audio_path)
    try:
        result = process_eremopedes_covilleae(audio_path)
        status, error = "ok", ""
    except Exception as e:
        result, status, error = {}, "error", str(e)
    eremopedes_rows.append({
        "species": "Eremopedes_covilleae",
        "file":    spec_path.name,
        "status":  status,
        "error":   error,
        **result,
    })

results_df = pd.read_csv(output_csv)
patched    = results_df[results_df["species"] != "Eremopedes_covilleae"].copy()
patched    = pd.concat([patched, pd.DataFrame(eremopedes_rows)], ignore_index=True)
patched    = patched[csv_columns]
patched.to_csv(output_csv, index=False)

print(f"Reprocessed {len(eremopedes_rows)} Eremopedes_covilleae file(s):")
for r in eremopedes_rows:
    print(f"  {r['file']}: {r}")


## Results

In [ ]:
df    = pd.read_csv(output_csv)
df_ok = df[df["status"] == "ok"].copy()
print(f"{len(df_ok)} images processed successfully")
df_ok[[
    "species", "element_length", "inter_element_interval",
    "inter_burst_interval", "elements_per_burst",
]]

## Merge with Webscraping Data

Joins the pipeline output to `katydid_df` (from `Webscraping.ipynb`) on the
oscillogram file ID. **Run `Webscraping.ipynb` at least once before this
cell.**

In [ ]:
import os
import re

# Load the webscraping DataFrame from IPython's persistent store.
# Requires that %store katydid_df was run in Webscraping.ipynb beforehand.
%store -r katydid_df


def spec_id_from_url(url):
    """'https://orthsoc.org/sina/010so.jpg' -> '010so'"""
    if pd.isna(url):
        return None
    return os.path.splitext(os.path.basename(str(url)))[0]


def spec_id_from_filename(filename):
    """'Amblycorypha_oblongifolia_spectrogram_010so.jpg' -> '010so'"""
    match = re.search(r'_spectrogram_(.+)\.[^.]+$', str(filename))
    return match.group(1) if match else None


def audio_id_from_url(url):
    """Audio-URL basename, e.g. '.../801-acrolophitus.mp3' -> '801-acrolophitus'.
    Preferred over spec_id for File_ID: many recordings share a generic
    'sound.gif' spectrogram, but Audio_Link basenames are distinct."""
    if pd.isna(url):
        return None
    return os.path.splitext(os.path.basename(str(url)))[0]


# Build join keys
# Note: katydid_df uses 'Description' (not 'Description of Whole Audio File')
webscraping_data = katydid_df[[
    'Species', 'Temperature (\u00b0C)', 'Location',
    'Description', 'Map', 'Spectrogram', 'Audio_Link',
]].copy()
webscraping_data['_spec_id'] = webscraping_data['Spectrogram'].apply(spec_id_from_url)
webscraping_data['_audio_id'] = webscraping_data['Audio_Link'].apply(audio_id_from_url)

# _spec_id isn't unique across species (many share a generic id like 'sound'),
# so the join also keys on _species_key--both sides normalized to
# lowercase-underscore--to keep 'sound' matching only within one species.
webscraping_data['_species_key'] = (
    webscraping_data['Species'].str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
)

processing_results = df_ok[[
    'species', 'file', 'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].copy()
processing_results['_spec_id'] = processing_results['file'].apply(spec_id_from_filename)
processing_results['_species_key'] = processing_results['species'].str.strip().str.lower()

# Join on oscillogram file ID *and* species, so a generic shared id like 'sound'
# only matches within the same species instead of across all species that used it.
merged = processing_results.merge(
    webscraping_data.drop(columns='Spectrogram'),
    on=['_spec_id', '_species_key'],
    how='left',
)

# Drop processed files with no matching metadata row (e.g. a duplicate
# '..._audio_sound.mp3' with no SINA spectrogram entry)--they'd show as a
# broken "nan / nan" species in the viewer.
unmatched = merged[merged['Species'].isna()]
if len(unmatched):
    print(f"Dropping {len(unmatched)} processed file(s) with no matching webscraping "
          f"metadata (species, spec_id): "
          + ", ".join(f"{r['species']}/{r['_spec_id']}" for _, r in unmatched.iterrows()))
merged = merged[merged['Species'].notna()].copy()

# Split 'Genus species' into two separate columns
merged[['Genus', 'Species']] = merged['Species'].str.split(' ', n=1, expand=True)

# Prefer the audio file's own id (traceable back to the actual source recording);
# fall back to the spectrogram-derived id for the rare rows with no Audio_Link.
merged['File_ID'] = merged['_audio_id'].fillna(merged['_spec_id'])

# Column names must match the viewer exactly. File_ID is the source-recording id
# (from Audio_Link, else spectrogram-derived); Spec_ID is kept separately
# because it--not File_ID--is embedded in the generated PNG filename the
# viewer matches on.
katydid_final = merged[[
    'Genus', 'Species',
    'Temperature (\u00b0C)', 'Location',
    'Description', 'Map',
    'File_ID', '_spec_id',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].rename(columns={
    'Temperature (\u00b0C)':   'Temperature',
    '_spec_id':                'Spec_ID',
    'element_length':          'Element_Length',
    'inter_element_interval':  'Inter-Element_Interval',
    'inter_burst_interval':    'Inter-Burst_Interval',
    'elements_per_burst':      'Elements_Per_Burst',
    'min_elements_per_burst':  'Min_Elements_Per_Burst',
    'max_elements_per_burst':  'Max_Elements_Per_Burst',
})

print(
    f'{len(katydid_final)} rows  |  '
    f'{katydid_final["Genus"].nunique()} genera  |  '
    f'{katydid_final["Species"].nunique()} species'
)
katydid_final.head()


In [ ]:
# Store for use by Display_Function.ipynb (%store -r katydid_final)
%store katydid_final

In [ ]:
katydid_final.to_csv(Path.home() / "Discrete_Signals" / "katydid_single_burst_type.csv", index=False)